In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("tested.csv")

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNulls per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

print("\nValue ranges (numeric columns):")
print(df.describe())

Shape: (418, 12)

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Nulls per column:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Duplicate rows: 0

Value ranges (numeric columns):
       PassengerId    Survived      Pclass         Age       SibSp  \
count   418.000000  418.000000  418.000000  332.000000  418.000000   
mean   1100.500000    0.363636    2.265550   30.272590    0.447368   
std     120.810458    0.481622    0.841838   14.181209    0.896760   
min     892.000000    0.000000    1.000000    0.170000    0.000000   
25%     996.2500

### Data Quality Report
I loaded the dataset and generated an initial report. Here's what I found:

- **Shape:** 418 rows, 12 columns
- **Duplicates:** None — 0 duplicate rows detected
- **Missing values:**
  - `Age`: 86 missing (about 21% of rows)
  - `Fare`: 1 missing (negligible)
  - `Cabin`: 327 missing (about 78% of rows — the vast majority of this column is empty)
- **Data type issues:** All columns currently have appropriate-looking types (numeric columns are int/float, text columns are object), so no immediate dtype conversion is needed except formatting checks I'll do later.
- **Value range anomalies:** 
  - `Age` ranges from 0.17 (an infant) to 76 — both plausible, no obvious errors.
  - `Fare` ranges from 0 to 512.33. A fare of 0 is unusual but plausible (crew members or comped tickets), while 512.33 is far higher than the 75th percentile (31.50) and looks like a genuine outlier I'll investigate later with the IQR method.
  - No negative values anywhere, which is a good sign — no obviously corrupted data.

In [9]:
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

median_fare = df['Fare'].median()
df['Fare'] = df['Fare'].fillna(median_fare)

df['Cabin'] = df['Cabin'].fillna('Unknown')

print("Remaining nulls:\n", df.isnull().sum())

Remaining nulls:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64


### Observation: Missing Data Handling
For `Age` and `Fare`, I used median imputation since both are numeric and 
right-skewed — median avoids being distorted by outliers the way mean would be. 
For `Cabin`, deleting rows wasn't an option since 78% of the column is missing 
— that would gut the dataset. Instead, I filled missing values with 'Unknown', 
treating the absence of a cabin record as its own category rather than guessing.

In [10]:
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]

print(f"Rows before: {before_rows}, Rows after: {after_rows}")
print(f"Duplicates removed: {before_rows - after_rows}")

Rows before: 418, Rows after: 418
Duplicates removed: 0


### Observation: Duplicate Removal
I checked for and removed duplicate rows. The result showed 0 duplicates 
in this dataset, so no rows were actually removed — but I ran the process 
formally to confirm the dataset is clean on this front rather than just 
assuming it.

In [12]:
embarked_map = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
df['Embarked'] = df['Embarked'].map(embarked_map)

print(df['Embarked'].unique())

['Queenstown' 'Southampton' 'Cherbourg']


### Observation: Standardization
I checked `Sex`, `Embarked`, and `Pclass` for inconsistent formatting (mixed 
casing, typos, abbreviations) and found none — all three were already clean. 
There's also no date column in this dataset, so date standardization doesn't 
apply here. 

As a genuine standardization step, I converted the `Embarked` column's single-
letter port codes (S, C, Q) into their full port names (Southampton, Cherbourg, 
Queenstown), making the data more readable and analysis-ready rather than 
relying on codes that require a lookup to interpret.

In [13]:
def find_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

age_outliers, age_low, age_high = find_outliers_iqr('Age')
fare_outliers, fare_low, fare_high = find_outliers_iqr('Fare')

print(f"Age outlier bounds: [{age_low:.2f}, {age_high:.2f}]")
print(f"Number of Age outliers: {len(age_outliers)}")

print(f"\nFare outlier bounds: [{fare_low:.2f}, {fare_high:.2f}]")
print(f"Number of Fare outliers: {len(fare_outliers)}")

Age outlier bounds: [3.88, 54.88]
Number of Age outliers: 36

Fare outlier bounds: [-27.47, 66.84]
Number of Fare outliers: 55


In [14]:
print("Sample of Age outliers:\n", age_outliers[['Name', 'Age']].head())
print("\nSample of Fare outliers:\n", fare_outliers[['Name', 'Fare']].head())

Sample of Age outliers:
                                                Name   Age
2                         Myles, Mr. Thomas Francis  62.0
13                             Howard, Mr. Benjamin  63.0
20                           Rothschild, Mr. Martin  55.0
48  Bucknell, Mrs. William Robert (Emma Eliza Ward)  60.0
69              Fortune, Mrs. Mark (Mary McDougald)  60.0

Sample of Fare outliers:
                                                Name      Fare
12    Snyder, Mrs. John Pillsbury (Nelle Stevenson)   82.2667
24  Ryerson, Mrs. Arthur Larned (Emily Maria Borie)  262.3750
48  Bucknell, Mrs. William Robert (Emma Eliza Ward)   76.2917
53                       Fortune, Miss. Ethel Flora  263.0000
59                      Chaudanson, Miss. Victorine  262.3750


### Observation: Outlier Detection
Using the IQR method, I found 36 outliers in Age (mostly very young children 
and passengers over 55) and 55 outliers in Fare (passengers who paid well 
above the typical fare). 

I decided to retain all of these rather than remove or cap them. Looking at 
the actual values, none of them look like data entry errors — a 76-year-old 
passenger or a $512 fare are both realistic given the Titanic had real elderly 
passengers and a genuine First Class with expensive tickets. Removing them 
would mean losing real, meaningful information rather than fixing a data 
quality problem.

In [15]:
df['PassengerId'] = df['PassengerId'].astype(str)
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

print(df.dtypes)

PassengerId      object
Survived          int64
Pclass         category
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Cabin            object
Embarked       category
dtype: object


### Observation: Data Type Correction
I converted `PassengerId` to a string since it's an identifier, not a value 
to do arithmetic on. I also converted `Pclass`, `Sex`, and `Embarked` to the 
`category` dtype since they're categorical variables with a small fixed set 
of values — this is more accurate than storing them as generic numbers or 
text, and more memory-efficient too. `Age`, `Fare`, and the count columns 
(`SibSp`, `Parch`, `Survived`) were already correctly typed as numeric.

In [16]:
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Null Count (Age)', 'Null Count (Fare)', 'Null Count (Cabin)',
               'Duplicate Rows', 'PassengerId dtype', 'Pclass dtype', 'Sex dtype'],
    'Before Cleaning': [418, 86, 1, 327, 0, 'int64', 'int64', 'object'],
    'After Cleaning': [df.shape[0], df['Age'].isnull().sum(), df['Fare'].isnull().sum(),
                       df['Cabin'].isnull().sum(), df.duplicated().sum(),
                       df['PassengerId'].dtype, df['Pclass'].dtype, df['Sex'].dtype]
})

print(summary)

               Metric Before Cleaning After Cleaning
0          Total Rows             418            418
1    Null Count (Age)              86              0
2   Null Count (Fare)               1              0
3  Null Count (Cabin)             327              0
4      Duplicate Rows               0              0
5   PassengerId dtype           int64         object
6        Pclass dtype           int64       category
7           Sex dtype          object       category


### Observation: Before vs. After Summary
This table makes the impact of the cleaning process clear: nulls in Age, Fare, 
and Cabin all went from real gaps to zero, dtypes for PassengerId, Pclass, and 
Sex are now semantically correct instead of generic, and the row count stayed 
at 418 throughout since we didn't need to drop any rows — every missing value 
was handled through imputation instead of deletion (except duplicates, of 
which there were none to begin with).

In [17]:
df.to_csv("titanic_cleaned.csv", index=False)
print("Saved as titanic_cleaned.csv")

Saved as titanic_cleaned.csv


## Conclusion
Starting from the raw Titanic dataset, I identified missing values in Age, 
Fare, and Cabin, and handled each with an approach suited to that column — 
median imputation for the numeric columns, and an explicit 'Unknown' category 
for Cabin rather than dropping the majority of the dataset. I confirmed there 
were no duplicate rows, standardized the Embarked column into readable port 
names, checked for outliers using the IQR method and chose to retain them 
since they represented real passengers rather than data errors, and corrected 
data types so identifiers and categorical fields are represented properly. 
The final cleaned dataset (`titanic_cleaned.csv`) has zero nulls, zero 
duplicates, and semantically correct data types, while preserving all 418 
original rows.